# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.json_normalize(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
unmatched = joined[joined['_merge'] == 'left_only']
print(unmatched['vendor_id'].unique())
print(unmatched['qty'].sum())
joined

['V-07']
3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
#  V-07 has no vendor record, so it gets labeled Unknown (V-07) instead of being dropped
# this keeps its 3 units visible and the total at 7
by_vendor = (
    joined.assign(name=joined['name'].fillna('Unknown (' + joined['vendor_id'] + ')'))
    .groupby('name')['qty']
    .sum()
)
by_vendor

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown (V-07),3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
by_zone = joined.groupby('zone')['qty'].sum()
print(by_zone)
print(by_zone.idxmax())
print(by_zone.sum())
print(joined.loc[joined['zone'].isna(), 'qty'].sum())

zone
A    4
Name: qty, dtype: int64
A
4
3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged = pd.json_normalize(
    [{**o, 'lines': o.get('lines', [])} for o in ragged_json],
    record_path='lines',
    meta=['order', 'vendor_id']
)
ragged['qty'] = ragged['qty'].astype('Int64')
ragged

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a: Each row holds exactly one item and its quantity, so any question about items or quantities becomes a simple groupby and sum. Total units per vendor, per zone, or per item (for example, how many Sodas sold across all orders). With one row per order, the quantities are stuck inside a nested list in each row, so you'd have to unpack them with a loop or apply before you could count anything.

b: A quantity of zero means we know nothing was sold. A missing quantity means we don't know how many were sold. If you fill missing values with 0, unknown data looks like a real zero. Totals come out too low with no warning, averages get pulled down, and you can no longer find the bad records to fix or report them. Data quality checks also stop catching the problem, because a filled in 0 looks like a valid value.